# 🔍 Regex/Parallel · 👁 Vision Power · δ Dirac Delta · 6-Vec · 🔬 ASIC Photonics · ⚡ EM · 🔢 Integer · 🎮 OOP Pygame

| § | Topic |
|---|---|
| §1 | Regex advanced + parallel corpus scan |
| §2 | Vision: max/min/average power, PSD, spatial frequency |
| §3 | Dirac delta — numerical approximations, SymPy, convolution identity |
| §4 | 6-vectors (screw theory) + GEMM benchmarking |
| §5 | ASIC photonics — MZI, ring resonator, layout-level simulation |
| §6 | Early electromagnetism — Coulomb → Gauss → Faraday → Maxwell |
| §7 | Integer calculations — GCD, modular arithmetic, fixed-point DSP |
| §8 | OOP pygame game engine — asset loader, entity/component system |

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import re, time, math, warnings
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from functools import lru_cache
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple
warnings.filterwarnings('ignore')
np.random.seed(2026)
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math
print('Regex - Vision - Dirac - 6Vec - ASIC - EM - Int - Pygame  ready')

---
## §1 🔍 Regex Advanced + Parallel Corpus Scan

**Regex engine internals:** NFA → DFA compilation. Python `re` uses backtracking NFA.
Key patterns for engineering text mining:

```python
r'(?P<name>\w+)\s*=\s*(?P<val>[\d.eE+-]+)\s*(?P<unit>[a-zA-Z/^]+)?'  # variable extraction
r'(?:(?<=\s)|^)(?:def|class)\s+(\w+)'                                  # code symbol scan
r'\b(?:TODO|FIXME|HACK|XXX)\b:?\s*(.+)'                                 # issue scanner
```

**Parallel scan:** split large text corpus into chunks, scan each chunk in a
`ThreadPoolExecutor` (I/O-bound), merge results.

In [ ]:
# §1 Regex advanced + parallel corpus scan

import re, time
from concurrent.futures import ThreadPoolExecutor
from collections import Counter, defaultdict

# §1.1 Named groups and pattern library
patterns = {
    'variable_assign':  re.compile(
        r'(?P<name>[A-Za-z_]\w*)\s*=\s*(?P<val>[+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)\s*'
        r'(?P<unit>[a-zA-Z_/^*]+)?'),
    'python_def':       re.compile(r'^\s*def\s+(?P<fname>\w+)\s*\((?P<args>[^)]*)\)', re.MULTILINE),
    'python_class':     re.compile(r'^\s*class\s+(?P<cname>\w+)(?:\((?P<bases>[^)]*)\))?', re.MULTILINE),
    'issue_tag':        re.compile(r'\b(?P<tag>TODO|FIXME|HACK|XXX|NOTE)\b:?\s*(?P<msg>.+)', re.IGNORECASE),
    'email':            re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Z|a-z]{2,}\b'),
    'ipv4':             re.compile(r'\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\b'),
    'semver':           re.compile(r'\bv?(?P<major>\d+)\.(?P<minor>\d+)\.(?P<patch>\d+)(?:-(?P<pre>[a-zA-Z0-9.]+))?\b'),
    'latex_eq':         re.compile(r'\$\$?(.+?)\$\$?', re.DOTALL),
    'hex_constant':     re.compile(r'\b0x(?P<val>[0-9A-Fa-f]+)\b'),
}

# Synthetic corpus
corpus_docs = [
    '''
    # Physics constants
    c = 2.998e8 m/s  # speed of light
    h = 6.626e-34 J*s  # Planck constant
    k_B = 1.381e-23 J/K  # Boltzmann

    def wave_number(freq, v=c):
        # TODO: add dispersion correction
        return 2 * pi * freq / v

    class PhotonicCrystal:
        # FIXME: bandgap calculation wrong for TM modes
        pass

    Contact: physics@jalalilab.ucla.edu
    Server: 192.168.1.42
    Version: v2.3.1-beta
    ''',
    '''
    E_field = 1.5e6 V/m
    lambda_pump = 1550e-9 m
    n_eff = 2.4  # no units

    class MachZehnder:
        pass

    class RingResonator:
        # NOTE: Q factor assumes ideal coupling
        pass

    def transfer_matrix(phi, t):
        # TODO: vectorize over wavelength array
        return np.array([[t, 1j*(1-t**2)**0.5],[1j*(1-t**2)**0.5, t]])

    admin@photonics.edu  test@192.168.0.1
    Hash: 0xDEADBEEF  0x1A2B3C4D
    Version: v1.0.0  v3.14.159-rc1
    ''',
    '''
    # ASIC photonics tape-out
    pitch = 0.25e-6 m/pixel
    waveguide_width = 450e-9 m
    coupling_gap = 200e-9 m

    def place_grating_coupler(x, y, angle=0.0):
        # HACK: angle clamped to +-45 for DRC
        pass

    class PDKLayer:
        pass

    Version: v4.2.0
    IP: 10.0.0.1
    ''',
]

# §1.2 Single-document scan
def scan_doc(doc: str, doc_id: int = 0) -> dict:
    results = {'doc_id': doc_id, 'matches': defaultdict(list)}
    for name, pat in patterns.items():
        for m in pat.finditer(doc):
            results['matches'][name].append({
                'groups': m.groupdict() if m.groupdict() else m.group(),
                'pos': m.start()
            })
    return results

r0 = scan_doc(corpus_docs[0], 0)
print('§1.2 Single document scan:')
for pat_name, matches in r0['matches'].items():
    if matches:
        print(f'  [{pat_name}] ({len(matches)} matches):')
        for m in matches[:2]:
            print(f'    {m["groups"]}')

# §1.3 Parallel corpus scan
print('\n§1.3 Parallel corpus scan:')
# Scale up corpus for timing
big_corpus = corpus_docs * 500   # 1500 docs

def scan_chunk(args):
    chunk, start_id = args
    return [scan_doc(doc, start_id+i) for i,doc in enumerate(chunk)]

CHUNK_SIZE = 100
chunks = [(big_corpus[i:i+CHUNK_SIZE], i) for i in range(0, len(big_corpus), CHUNK_SIZE)]

t0 = time.time()
serial_results = [scan_doc(doc,i) for i,doc in enumerate(big_corpus)]
dt_serial = time.time()-t0

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    parallel_results = list(ex.map(scan_chunk, chunks))
dt_parallel = time.time()-t0
parallel_flat = [r for chunk in parallel_results for r in chunk]

print(f'  {len(big_corpus)} docs: serial={dt_serial*1000:.1f}ms  parallel(8T)={dt_parallel*1000:.1f}ms  speedup={dt_serial/dt_parallel:.1f}x')

# §1.4 Regex performance: compiled vs inline
pattern_str = r'\b[A-Za-z_]\w*\s*=\s*[\d.eE+-]+'
pat_compiled = re.compile(pattern_str)
test_text    = '\n'.join(corpus_docs) * 20

t0 = time.time()
for _ in range(500): re.findall(pattern_str, test_text)
dt_inline = time.time()-t0

t0 = time.time()
for _ in range(500): pat_compiled.findall(test_text)
dt_compiled = time.time()-t0

print(f'\n§1.4 Regex performance (500 iterations):')
print(f'  Inline re.findall:   {dt_inline*1000:.1f}ms')
print(f'  Compiled pattern:    {dt_compiled*1000:.1f}ms  ({dt_inline/dt_compiled:.1f}x speedup)')

# §1.5 Aggregate statistics from parallel results
all_counts = Counter()
for doc_r in serial_results:
    for pat_name, matches in doc_r['matches'].items():
        all_counts[pat_name] += len(matches)

print('\n§1.5 Corpus aggregate match counts (1500 docs):')
for pat_name, count in sorted(all_counts.items(), key=lambda x:-x[1]):
    print(f'  {pat_name:20s}: {count:6d}')

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(16,4))
# Match frequency bar
names_p = list(all_counts.keys()); vals_p = [all_counts[n] for n in names_p]
axes[0].barh(names_p, vals_p, color=plt.cm.tab10(np.linspace(0,0.9,len(names_p))), alpha=0.85)
axes[0].set_xlabel('Total matches'); axes[0].set_title('§1.5 Regex match frequency\nacross 1500 docs')

# Speedup vs workers
workers_list = [1,2,4,8]
speedups = []
for nw in workers_list:
    t0=time.time()
    with ThreadPoolExecutor(max_workers=nw) as ex:
        list(ex.map(scan_chunk, chunks))
    speedups.append(dt_serial/(time.time()-t0))
axes[1].plot(workers_list, speedups, 'o-', lw=2.5, color='royalblue', ms=8)
axes[1].plot(workers_list, workers_list, 'k--', lw=1.5, alpha=0.5, label='Ideal linear')
axes[1].set_xlabel('Workers'); axes[1].set_ylabel('Speedup vs serial')
axes[1].set_title('§1.3 Parallel scan speedup\nvs number of threads')
axes[1].legend(fontsize=9)

# NFA state diagram (schematic)
ax3 = axes[2]; ax3.axis('off')
ax3.set_title('§1 Regex NFA structure\n(\\d+\\.\\d* matching)', fontsize=10)
states_nfa = [(0.1,0.5,'q0\nstart'),(0.35,0.5,'q1\n\\d+'),(0.6,0.5,'q2\n\\.'),(0.85,0.5,'q3\n\\d*')]
for x,y,lbl in states_nfa:
    ax3.add_patch(plt.Circle((x,y),0.09,facecolor='#3498db',edgecolor='white',lw=2,alpha=0.85))
    ax3.text(x,y,lbl,ha='center',va='center',fontsize=8,color='white',fontweight='bold')
for i in range(len(states_nfa)-1):
    x1,y1,_ = states_nfa[i]; x2,y2,_ = states_nfa[i+1]
    ax3.annotate('',xy=(x2-0.09,y2),xytext=(x1+0.09,y1),
                 arrowprops=dict(arrowstyle='-|>',color='#333',lw=2))
ax3.add_patch(plt.Circle((0.85,0.5),0.11,fill=False,edgecolor='gold',lw=3))
ax3.text(0.5,0.1,'NFA: O(n) via Thompson construction\n(Python re uses backtracking: worst case O(2^n))',
         ha='center',fontsize=8,color='#555')
plt.suptitle('§1: Regex Advanced + Parallel Corpus Scan',fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 👁 Vision: Max/Min/Average Power · PSD · Spatial Frequency

**Image power** = sum of squared pixel intensities (Parseval's theorem connects
spatial-domain power to frequency-domain power):
$$P = \frac{1}{MN}\sum_{m,n}|I(m,n)|^2 = \frac{1}{MN}\sum_{u,v}|\hat{I}(u,v)|^2$$

**Max vision** = highest spatial frequency resolved = Nyquist limit = $f_s/2$
(detector pixel pitch determines cut-off).

**Min vision** = DC component (zero frequency) = mean intensity.

**Average power** = mean square value = second moment of intensity distribution.

**PSD** (Power Spectral Density) from 2D FFT:
$S(u,v) = |\mathcal{F}\{I\}|^2 / (MN)$

In [ ]:
# §2 Vision power analysis

# §2.1 Synthetic image suite
N_img = 256
x_im  = np.linspace(0,1,N_img)
y_im  = np.linspace(0,1,N_img)
X_im,Y_im = np.meshgrid(x_im,y_im)

# Generate test images
images = {}
# Natural image (1/f^2 PSD)
noise_white = np.random.randn(N_img,N_img)
F_white     = np.fft.fft2(noise_white)
u_f = np.fft.fftfreq(N_img); v_f = np.fft.fftfreq(N_img)
UF,VF = np.meshgrid(u_f,v_f); freq_2d = np.sqrt(UF**2+VF**2)
freq_2d[0,0] = 1e-10
filter_1f2 = 1.0/freq_2d
images['Natural (1/f^2)'] = np.real(np.fft.ifft2(F_white*filter_1f2))

# Sinusoidal grating
f_grating = 12   # cycles per image
images['Grating 12 cyc'] = np.sin(2*np.pi*f_grating*X_im)

# Gaussian blob
images['Gaussian blob'] = np.exp(-50*((X_im-0.5)**2+(Y_im-0.5)**2))

# Random white noise
images['White noise'] = np.random.randn(N_img,N_img)

# Checkerboard
chk = np.zeros((N_img,N_img))
chk[::16,::16] = 1; chk[8::16,8::16] = 1
images['Checkerboard'] = chk

# §2.2 Power statistics for each image
print('§2.2 Image power statistics:')
print(f'  {"Image":20s} {"Min":8s} {"Max":8s} {"Mean":8s} {"Avg Power":10s} {"PSD Peak":10s}')
print('  '+'-'*68)
img_stats = {}
for name, Im in images.items():
    Im_norm = Im / (Im.std()+1e-10)   # normalize
    F2d     = np.fft.fftshift(np.fft.fft2(Im_norm))
    PSD     = (np.abs(F2d)**2)/(N_img**2)
    avg_pow = np.mean(Im_norm**2)
    psd_peak= PSD.max()
    # Spatial frequency at PSD peak (excluding DC)
    PSD_nodc= PSD.copy(); PSD_nodc[N_img//2,N_img//2]=0
    peak_idx= np.unravel_index(PSD_nodc.argmax(), PSD_nodc.shape)
    f_peak  = np.sqrt((peak_idx[0]-N_img//2)**2+(peak_idx[1]-N_img//2)**2)/N_img
    img_stats[name] = {'Im':Im_norm,'PSD':PSD,'avg_pow':avg_pow,'f_peak':f_peak}
    print(f'  {name:20s} {Im_norm.min():8.3f} {Im_norm.max():8.3f} {Im_norm.mean():8.4f} '
          f'{avg_pow:10.4f} {f_peak:10.4f} cyc/px')

# §2.3 Parseval's theorem verification
print('\n§2.3 Parseval theorem check (spatial vs frequency domain power):')
for name, st in img_stats.items():
    Im_n  = st['Im']
    P_spa = np.mean(Im_n**2)
    P_fre = np.sum(st['PSD'])/(N_img**2) * (N_img**2)
    P_fre2= np.mean(np.abs(np.fft.fft2(Im_n))**2)/N_img**2
    print(f'  {name:20s}: P_spatial={P_spa:.6f}  P_Parseval={P_fre2:.6f}  match={abs(P_spa-P_fre2)<1e-6}')

# §2.4 Radially-averaged PSD (isotropic)
def radial_psd(PSD):
    cy,cx = N_img//2, N_img//2
    y_r,x_r = np.ogrid[-cy:N_img-cy, -cx:N_img-cx]
    r    = np.sqrt(x_r**2+y_r**2).astype(int)
    r_max= r.max()
    psd_r= np.array([PSD[r==ri].mean() if (r==ri).any() else 0 for ri in range(r_max+1)])
    f_r  = np.arange(r_max+1)/N_img
    return f_r, psd_r

# ── Plots ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(18,10))
gs  = gridspec.GridSpec(3,5,fig,hspace=0.4,wspace=0.3)

for col_i,(name,st) in enumerate(img_stats.items()):
    ax_im  = fig.add_subplot(gs[0,col_i])
    ax_psd = fig.add_subplot(gs[1,col_i])
    Im_n   = st['Im']
    PSD    = st['PSD']
    ax_im.imshow(Im_n, cmap='gray', aspect='equal')
    ax_im.axis('off'); ax_im.set_title(name,fontsize=8)
    ax_psd.imshow(np.log1p(PSD), cmap='hot', aspect='equal')
    ax_psd.axis('off'); ax_psd.set_title(f'PSD log\nP={st["avg_pow"]:.3f}',fontsize=8)

# Radial PSD comparison
ax_rad = fig.add_subplot(gs[2,:3])
for name,st in img_stats.items():
    f_r,psd_r = radial_psd(st['PSD'])
    ax_rad.semilogy(f_r[1:], psd_r[1:]+1e-12, lw=2, label=name)
# Reference slopes
f_ref = np.linspace(0.01,0.5,100)
ax_rad.semilogy(f_ref, 0.05*f_ref**(-2), 'k--', lw=1.5, label='f^-2 (natural)')
ax_rad.semilogy(f_ref, 0.001*f_ref**(-4)*0.1, 'k:', lw=1.5, label='f^-4')
ax_rad.set_xlabel('Spatial frequency (cycles/pixel)')
ax_rad.set_ylabel('Radial PSD')
ax_rad.set_title('§2.4 Radially-averaged PSD\n(isotropic assumption)')
ax_rad.legend(fontsize=8)

# Power budget table
ax_tbl = fig.add_subplot(gs[2,3:])
ax_tbl.axis('off')
tbl_data = [[n, f'{s["avg_pow"]:.4f}', f'{s["f_peak"]:.4f}']
            for n,s in img_stats.items()]
tbl = ax_tbl.table(cellText=tbl_data,
                    colLabels=['Image','Avg Power','Peak freq'],
                    loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1,1.6)
ax_tbl.set_title('§2 Power budget\nsummary', fontsize=10)

plt.suptitle('§2: Vision Power — Max/Min/Avg · PSD · Parseval · Radial Spectrum',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 δ Dirac Delta — Numerical Approx · SymPy · Convolution Identity

**Dirac delta** is a distribution, not a function. Key properties:
$$\int_{-\infty}^{\infty} \delta(x-a)\,f(x)\,dx = f(a) \qquad \text{(sifting property)}$$
$$\delta(ax) = \frac{1}{|a|}\delta(x) \qquad \delta(f(x)) = \sum_i \frac{\delta(x-x_i)}{|f'(x_i)|}$$

**Numerical approximations** (all normalized to unit area):
- Gaussian: $\delta_\varepsilon(x) = \frac{1}{\varepsilon\sqrt{\pi}}e^{-x^2/\varepsilon^2}$
- sinc: $\delta_\varepsilon(x) = \frac{\sin(x/\varepsilon)}{\pi x}$
- Triangle: $\delta_\varepsilon(x) = \max(0, (1-|x|/\varepsilon)/\varepsilon)$
- Derivative of Heaviside: $\delta(x) = H'(x)$

**Convolution identity:** $f * \delta = f$ — fundamental to LTI systems.

In [ ]:
# §3 Dirac delta

import sympy as sp
import numpy as np

x_d, eps_d, a_d, t_d = sp.symbols('x epsilon a t', real=True)

# §3.1 SymPy DiracDelta
print('§3.1 SymPy DiracDelta sifting property:')
integrals_dd = [
    (sp.DiracDelta(x_d-2)*sp.sin(x_d),    (x_d,-sp.oo,sp.oo), 'int sin(x)*delta(x-2) dx'),
    (sp.DiracDelta(x_d)*sp.exp(x_d**2),   (x_d,-sp.oo,sp.oo), 'int exp(x^2)*delta(x) dx'),
    (sp.DiracDelta(x_d-a_d)*x_d**2,       (x_d,-sp.oo,sp.oo), 'int x^2*delta(x-a) dx'),
    (sp.DiracDelta(3*x_d+1),              None,                 'delta(3x+1) simplify'),
    (sp.DiracDelta(x_d**2-4),             None,                 'delta(x^2-4) simplify'),
]
for entry in integrals_dd:
    expr, bounds, name = entry
    if bounds:
        result = sp.integrate(expr, bounds)
        print(f'  {name}: {result}')
    else:
        result = sp.simplify(expr.rewrite(sp.DiracDelta))
        print(f'  {name}: {expr} -> {result}')

# §3.2 Numerical approximations
print('\n§3.2 Numerical delta approximations (sifting test):')
x_num = np.linspace(-5, 5, 10001); dx_n = x_num[1]-x_num[0]
a_test = 1.5   # sifting point
f_test = np.exp(-0.3*x_num**2)*np.cos(x_num)   # test function
f_at_a = np.exp(-0.3*a_test**2)*np.cos(a_test)

def delta_gauss(x, eps):
    return np.exp(-(x/eps)**2)/(eps*np.sqrt(np.pi))

def delta_lorentz(x, eps):
    return eps/(np.pi*(x**2+eps**2))

def delta_sinc(x, eps):
    arg = x/eps; arg[np.abs(arg)<1e-12]=1e-12
    return np.sin(np.pi*arg)/(np.pi*x+1e-15)

def delta_triangle(x, eps):
    return np.maximum(0, (1-np.abs(x)/eps)/eps)

def delta_regularized(x, eps):
    # Derivative of logistic function (sigmoid)
    s = 1/(1+np.exp(-x/eps))
    return s*(1-s)/eps

deltas = {
    'Gaussian':    delta_gauss,
    'Lorentzian':  delta_lorentz,
    'Triangle':    delta_triangle,
    'Regularized': delta_regularized,
}

eps_vals = [0.5, 0.1, 0.02]
print(f'  f(a={a_test}) exact = {f_at_a:.8f}')
for eps in eps_vals:
    print(f'\n  eps={eps}:')
    for name, dfn in deltas.items():
        d_vals = dfn(x_num-a_test, eps)
        sift   = np.trapz(f_test*d_vals, x_num)
        err    = abs(sift-f_at_a)
        print(f'    {name:12s}: integral={sift:.8f}  error={err:.2e}')

# §3.3 Convolution identity f*delta = f
print('\n§3.3 Convolution identity: f * delta = f')
f_conv    = np.where(np.abs(x_num)<2, np.cos(np.pi*x_num/4), 0.0)
delta_narrow = delta_gauss(x_num, 0.05)
# Discrete convolution
conv_result = np.convolve(f_conv, delta_narrow, mode='same')*dx_n
err_conv    = np.max(np.abs(conv_result - f_conv))
print(f'  max|f*delta - f| = {err_conv:.6f}  (eps=0.05, N={len(x_num)})')

# §3.4 Green's function connection
# For -d^2u/dx^2 = f(x) with u(0)=u(L)=0
# G(x,xi) = x*(L-xi)/L for x<xi  else xi*(L-x)/L
print('\n§3.4 Green function for -u\'\'=f (Poisson 1D):')
L_gf = 1.0; xi_vals = [0.25, 0.5, 0.75]
x_gf = np.linspace(0, L_gf, 500)
for xi_gf in xi_vals:
    G = np.where(x_gf <= xi_gf,
                 x_gf*(L_gf-xi_gf)/L_gf,
                 xi_gf*(L_gf-x_gf)/L_gf)
    # Verify: -G''(x,xi) = delta(x-xi)
    G_pp = -np.gradient(np.gradient(G, x_gf), x_gf)
    norm = np.trapz(G_pp, x_gf)
    print(f'  xi={xi_gf}: int(-G\'\'(x,xi))dx = {norm:.4f}  (should be 1.0)')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,4,figsize=(16,8))

# Delta approximations
x_plot = np.linspace(-2,2,1000)
for i,(name,dfn) in enumerate(deltas.items()):
    ax = axes[0][i]
    for eps,col in [(0.5,'red'),(0.2,'orange'),(0.05,'blue')]:
        d_p = dfn(x_plot, eps)
        ax.plot(x_plot, d_p, color=col, lw=2, label=f'eps={eps}')
    ax.set_title(f'§3.2 {name}\ndelta_eps(x)')
    ax.legend(fontsize=7); ax.set_xlabel('x'); ax.set_ylim(-0.5, 20)

# Sifting error vs eps
eps_range = np.logspace(-2, 0, 30)
ax_err = axes[1][0]
for name,dfn in deltas.items():
    errs = []
    for eps in eps_range:
        d_e = dfn(x_num-a_test, eps)
        s   = np.trapz(f_test*d_e, x_num)
        errs.append(abs(s-f_at_a))
    ax_err.loglog(eps_range, errs, lw=2, label=name)
ax_err.set_xlabel('eps'); ax_err.set_ylabel('|sifting error|')
ax_err.set_title('§3.2 Sifting error\nvs approximation width')
ax_err.legend(fontsize=7)

# Convolution identity
axes[1][1].plot(x_num, f_conv, 'royalblue', lw=2.5, label='f(x)')
axes[1][1].plot(x_num, conv_result, 'r--', lw=2, label='f * delta_eps')
axes[1][1].set_xlim(-3,3); axes[1][1].legend(fontsize=9)
axes[1][1].set_title(f'§3.3 Convolution identity\nf * delta = f  (err={err_conv:.4f})')

# Green function
ax_gf = axes[1][2]
for xi_gf,col in zip(xi_vals,['red','green','blue']):
    G = np.where(x_gf<=xi_gf, x_gf*(L_gf-xi_gf)/L_gf, xi_gf*(L_gf-x_gf)/L_gf)
    ax_gf.plot(x_gf,G,color=col,lw=2,label=f'G(x,xi={xi_gf})')
ax_gf.set_xlabel('x'); ax_gf.set_ylabel('G(x,xi)')
ax_gf.set_title('§3.4 Green function\n-u\'\'=delta(x-xi), u(0)=u(1)=0')
ax_gf.legend(fontsize=8)

# Derivative of Heaviside
x_H   = np.linspace(-3,3,2000)
H_x   = (x_H > 0).astype(float)
dH    = np.gradient(H_x, x_H)
delta_g05 = delta_gauss(x_H, 0.1)
axes[1][3].plot(x_H, H_x, 'royalblue', lw=2, label='H(x)')
axes[1][3].plot(x_H, dH,  'red',       lw=2, label="H'(x) = delta")
axes[1][3].plot(x_H, delta_g05,'green',lw=1.5,ls='--',label='Gaussian delta')
axes[1][3].set_xlim(-2,2); axes[1][3].legend(fontsize=8)
axes[1][3].set_title('§3 Delta = dH/dx\nHeaviside derivative')

plt.suptitle('§3: Dirac Delta — Approx · Sifting · Convolution Identity · Green Function',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 🤖 6-Vectors (Screw Theory) + GEMM Benchmarking

**6-vector (twist/wrench)** in robotics and mechanics:
- **Twist:** $\mathcal{V} = (\omega, v) \in \mathbb{R}^6$ — angular + linear velocity
- **Wrench:** $\mathcal{F} = (\tau, f) \in \mathbb{R}^6$ — torque + force
- Power: $P = \mathcal{F}^T \mathcal{V}$
- **Adjoint transform:** $\mathcal{V}_{B} = [\text{Ad}_{T_{BA}}]\mathcal{V}_{A}$
  where $T \in SE(3)$ is a rigid body transform.

**GEMM (General Matrix-Matrix Multiply):** the workhorse of ML/physics simulation.
Peak FLOPs = $2MNK$ for $M\times K$ times $K\times N$.

In [ ]:
# §4 6-vectors (screw theory) + GEMM

import numpy as np
import time

# §4.1 SE(3) rigid body transforms
def skew(w):
    '''Skew-symmetric matrix [w]_x for w in R^3.'''
    return np.array([[0,-w[2],w[1]],[w[2],0,-w[0]],[-w[1],w[0],0]])

def exp_so3(w_vec):
    '''Rodrigues formula: exp([w]) = I + sin(theta)/theta*[w] + ...'''
    theta = np.linalg.norm(w_vec)
    if theta < 1e-10: return np.eye(3)
    w_hat = w_vec/theta
    K = skew(w_hat)
    return np.eye(3) + np.sin(theta)*K + (1-np.cos(theta))*K@K

def se3_from_rt(R, p):
    '''4x4 homogeneous transform T = [R p; 0 1].'''
    T = np.eye(4); T[:3,:3] = R; T[:3,3] = p; return T

def adjoint_T(T):
    '''6x6 adjoint matrix from 4x4 SE(3) transform.'''
    R = T[:3,:3]; p = T[:3,3]
    Ad = np.zeros((6,6))
    Ad[:3,:3] = R
    Ad[3:,3:] = R
    Ad[3:,:3] = skew(p)@R
    return Ad

# Example: robot link transforms
print('§4.1 SE(3) and 6-vector screw theory:')
# Link 1: rotation about z by 45 deg, translation (1,0,0)
theta1 = np.radians(45)
w1     = np.array([0,0,1])*theta1
R1     = exp_so3(w1)
T1     = se3_from_rt(R1, np.array([1.0,0,0]))
Ad1    = adjoint_T(T1)
print(f'  T_1 (R, p=[1,0,0], theta=45deg):')
print(f'  R =\n{R1.round(4)}')

# Twist in frame 0: pure rotation about z at 1 rad/s
V0 = np.array([0,0,1, 0,0,0])   # [omega; v]
V1 = Ad1 @ V0
print(f'  Twist in frame 0: {V0}')
print(f'  Twist in frame 1: {V1.round(4)}')

# §4.2 Wrench transformation
F0 = np.array([0,0,5, 10,0,0])   # [torque; force]
# Wrench transforms with inverse adjoint transpose
Ad1_inv_T = np.linalg.inv(Ad1).T
F1 = Ad1_inv_T @ F0
print(f'\n  Wrench in frame 0: {F0}')
print(f'  Wrench in frame 1: {F1.round(4)}')
P0 = F0 @ V0; P1 = F1 @ V1
print(f'  Power invariance: P0={P0:.4f}  P1={P1:.4f}  match={abs(P0-P1)<1e-8}')

# §4.3 Spatial inertia (6x6 mass matrix)
def spatial_inertia(m, I_cm, p_cm):
    '''6x6 spatial inertia matrix I_s = [I_o, m*p^; -m*p^, mI].'''
    p_sk = skew(p_cm)
    I_s  = np.zeros((6,6))
    I_s[:3,:3] = I_cm + m*(p_sk@p_sk.T)   # parallel axis theorem
    I_s[:3,3:] = m*p_sk
    I_s[3:,:3] = -m*p_sk
    I_s[3:,3:] = m*np.eye(3)
    return I_s

m_link = 1.5   # kg
I_cm   = np.diag([0.1, 0.2, 0.15])   # kg*m^2 principal inertias
p_cm   = np.array([0.3, 0, 0])
I_s    = spatial_inertia(m_link, I_cm, p_cm)
print(f'\n§4.2 Spatial inertia matrix (6x6):\n{I_s.round(4)}')

# §4.4 GEMM benchmarking
print('\n§4.3 GEMM benchmarking:')
sizes = [64, 128, 256, 512, 1024, 2048]
dtypes_bench = [('float32',np.float32), ('float64',np.float64)]

gemm_results = {}
for dtype_name, dtype in dtypes_bench:
    times_list = []
    flops_list = []
    for N in sizes:
        A = np.random.randn(N,N).astype(dtype)
        B = np.random.randn(N,N).astype(dtype)
        # Warmup
        _ = A@B
        # Time
        n_reps = max(1, int(1e9/(2*N**3)))
        t0 = time.time()
        for _ in range(n_reps): C = A@B
        dt_g = (time.time()-t0)/n_reps
        flops = 2*N**3
        gflops = flops/dt_g/1e9
        times_list.append(dt_g*1000)
        flops_list.append(gflops)
        print(f'  {dtype_name} N={N:5d}: {dt_g*1000:8.3f}ms  {gflops:7.2f} GFLOP/s')
    gemm_results[dtype_name] = (sizes, times_list, flops_list)

# §4.5 Strassen algorithm (recursive, educational)
def strassen(A, B):
    '''Strassen matrix multiply — O(n^2.807) vs O(n^3).'''
    n = len(A)
    if n <= 64: return A@B   # base case
    h = n//2
    A11,A12 = A[:h,:h], A[:h,h:]
    A21,A22 = A[h:,:h], A[h:,h:]
    B11,B12 = B[:h,:h], B[:h,h:]
    B21,B22 = B[h:,:h], B[h:,h:]
    M1 = strassen(A11+A22, B11+B22)
    M2 = strassen(A21+A22, B11)
    M3 = strassen(A11, B12-B22)
    M4 = strassen(A22, B21-B11)
    M5 = strassen(A11+A12, B22)
    M6 = strassen(A21-A11, B11+B12)
    M7 = strassen(A12-A22, B21+B22)
    C  = np.zeros((n,n))
    C[:h,:h] = M1+M4-M5+M7
    C[:h,h:] = M3+M5
    C[h:,:h] = M2+M4
    C[h:,h:] = M1-M2+M3+M6
    return C

N_st = 128
A_st = np.random.randn(N_st,N_st); B_st = np.random.randn(N_st,N_st)
t0 = time.time(); C_st = strassen(A_st,B_st); dt_st = time.time()-t0
t0 = time.time(); C_np = A_st@B_st; dt_np = time.time()-t0
err_st = np.max(np.abs(C_st-C_np))
print(f'\n§4.4 Strassen N={N_st}: dt={dt_st*1000:.2f}ms  numpy={dt_np*1000:.2f}ms  err={err_st:.2e}')
print(f'  (numpy wins via BLAS — Strassen advantage only at very large N)')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1,4,figsize=(16,5))

# GEMM throughput
ax1 = axes[0]
for dtype_name,(sizes_g,_,gflops_g) in gemm_results.items():
    ax1.loglog(sizes_g, gflops_g, 'o-', lw=2.5, ms=8, label=dtype_name)
ax1.set_xlabel('Matrix size N'); ax1.set_ylabel('GFLOP/s')
ax1.set_title('§4.3 GEMM throughput\n(np BLAS backend)')
ax1.legend(fontsize=9)

# Roofline (memory vs compute bound)
ax2 = axes[1]
I_range  = np.logspace(-2,3,200)   # arithmetic intensity (FLOPs/byte)
bw_GBs   = 30.0   # memory bandwidth GB/s
peak_TF  = 0.5    # peak FP32 TFLOP/s
roof_mem = bw_GBs * I_range/1e3   # TFLOP/s (memory bound)
roof_comp= np.full_like(I_range, peak_TF)
ax2.loglog(I_range, np.minimum(roof_mem,roof_comp), 'k', lw=2.5, label='Roofline')
ax2.fill_between(I_range, np.minimum(roof_mem,roof_comp), alpha=0.15, color='blue')
# Plot operations
ops_pts = {
    'DGEMM 1024': (2*1024**3/(1024**2*8*2)*1e-9, gemm_results['float64'][2][4]/1e3),
    'Vec dot N=1k':(2000/8000, 1e-4),
    'Conv 3x3':   (18/(9*4+4), 0.02),
}
for name,(I_op,perf_op) in ops_pts.items():
    ax2.scatter([I_op],[perf_op],s=100,zorder=5)
    ax2.text(I_op*1.2,perf_op,name,fontsize=7)
ax2.set_xlabel('Arithmetic intensity (FLOPs/byte)')
ax2.set_ylabel('Attainable TFLOP/s')
ax2.set_title('§4 Roofline model\ncompute vs memory bound')
ax2.legend(fontsize=8)

# 6-vector Lie algebra visualization
ax3 = axes[2]; ax3.axis('off')
ax3.set_title('§4.1 SE(3) twist/wrench\n6-vector structure', fontsize=10)
items_6v = [
    (r'Twist V = [omega; v]', 'angular+linear velocity', '#3498db'),
    (r'Wrench F = [tau; f]',  'torque+force',            '#e74c3c'),
    (r'Ad_T: R^6 -> R^6',    'frame change',             '#27ae60'),
    (r'Power P = F^T V',      'invariant scalar',         '#9b59b6'),
    (r'Inertia I_s 6x6',      'mass matrix in 6D',        '#e67e22'),
    (r'Euler: I_s V_dot = F', 'Newton-Euler EOM',         '#1abc9c'),
]
for i,(lhs,rhs,col) in enumerate(items_6v):
    y = 0.88-i*0.15
    ax3.add_patch(__import__('matplotlib.patches',fromlist=['FancyBboxPatch'])
                  .FancyBboxPatch((0.02,y-0.07),0.95,0.12,
                                   boxstyle='round,pad=0.01',facecolor=col,
                                   edgecolor='white',lw=1.5,alpha=0.85))
    ax3.text(0.50,y,f'{lhs}\n{rhs}',ha='center',va='center',
             fontsize=8,color='white',fontweight='bold')

# Spatial inertia heatmap
ax4 = axes[3]
im4 = ax4.imshow(np.abs(I_s), cmap='YlOrRd', aspect='auto')
ax4.set_title('§4.2 Spatial inertia\n|I_s| 6x6 matrix')
ax4.set_xlabel('column'); ax4.set_ylabel('row')
plt.colorbar(im4, ax=ax4)
for i in range(6):
    for j in range(6):
        ax4.text(j,i,f'{I_s[i,j]:.2f}',ha='center',va='center',fontsize=6,
                 color='white' if abs(I_s[i,j])>0.3 else 'black')

plt.suptitle('§4: 6-Vectors (Screw Theory) + GEMM · SE(3) · Spatial Inertia · Roofline',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §5 🔬 ASIC Photonics — MZI · Ring Resonator · Transfer Matrix · Layout

**Silicon photonics ASIC:** passive and active optical components integrated
on a Si or SiN chip. Key building blocks:

| Component | Function | Transfer matrix |
|-----------|----------|-----------------|
| **Directional coupler** | 50:50 or T:R splitter | $\begin{pmatrix}t & j\kappa\\j\kappa & t\end{pmatrix}$ |
| **Phase shifter** | Thermo-optic / PN | $\begin{pmatrix}e^{j\phi} & 0\\0 & 1\end{pmatrix}$ |
| **MZI** | coupler + phase + coupler | product of above |
| **Ring resonator** | frequency-selective drop | Lorentzian transmission |

**Add-drop ring:** $T_{thru} = \frac{t_2 - t_1 e^{j\phi}}{1 - t_1 t_2 e^{j\phi}}$
FSR $= c/(n_g L)$, $Q = \lambda_0/\Delta\lambda$.

In [ ]:
# §5 ASIC photonics — MZI and ring resonator simulation

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# §5.1 Transfer matrix formalism
def coupler_tm(t, kappa=None):
    '''2x2 transfer matrix for a directional coupler.'''
    if kappa is None: kappa = np.sqrt(1-t**2)
    return np.array([[t, 1j*kappa],[1j*kappa, t]])

def phase_tm(phi):
    '''Phase shifter transfer matrix (upper arm).'''
    return np.array([[np.exp(1j*phi), 0],[0, 1.0+0j]])

def mzi_tm(phi, t1=1/np.sqrt(2), t2=1/np.sqrt(2)):
    '''MZI: DC1 -> phase shifter -> DC2.'''
    C1 = coupler_tm(t1)
    P  = phase_tm(phi)
    C2 = coupler_tm(t2)
    return C2 @ P @ C1

# MZI transmission vs phase
phi_arr = np.linspace(0, 4*np.pi, 1000)
T_bar   = np.zeros(len(phi_arr))   # bar port
T_cross = np.zeros(len(phi_arr))   # cross port
for i,phi in enumerate(phi_arr):
    M = mzi_tm(phi)
    # Input: [1, 0] (only port 1 excited)
    out = M @ np.array([1.0+0j, 0.0+0j])
    T_bar[i]   = np.abs(out[0])**2
    T_cross[i] = np.abs(out[1])**2

print('§5.1 MZI transfer matrix:')
print(f'  T_bar + T_cross = {(T_bar+T_cross).mean():.6f} (energy conservation)')
print(f'  Extinction ratio (bar): {10*np.log10(T_bar.max()/T_bar.min()):.1f} dB')

# §5.2 Ring resonator (add-drop)
print('\n§5.2 Ring resonator (add-drop):')
lambda_c = 1550e-9   # center wavelength m
n_g      = 3.8       # group index Si at 1550nm
L_ring   = 50e-6     # ring circumference 50 um
FSR      = lambda_c**2 / (n_g * L_ring) * 1e9  # nm
print(f'  FSR = {FSR:.2f} nm  (L={L_ring*1e6:.0f}um, n_g={n_g})')

# Add-drop ring transmission
t1_ring  = 0.95    # through coupling
t2_ring  = 0.95    # drop coupling
alpha_dB = 3.0     # ring loss dB/cm
alpha    = 10**(alpha_dB/10 * L_ring*100)   # round-trip power loss factor
a        = np.sqrt(1/alpha)               # field amplitude after round trip

# Wavelength sweep around resonance
dlambda  = np.linspace(-3e-9, 3e-9, 5000)
lambda_sw= lambda_c + dlambda
phi_rt   = 2*np.pi*n_g*L_ring/lambda_sw   # round-trip phase

# Transfer functions
T_thru   = (t1_ring**2 - 2*t1_ring*t2_ring*a*np.cos(phi_rt) + (t2_ring*a)**2) / \
            (1 - 2*t1_ring*t2_ring*a*np.cos(phi_rt) + (t1_ring*t2_ring*a)**2)
T_drop   = ((1-t1_ring**2)*(1-t2_ring**2)*a) / \
            (1 - 2*t1_ring*t2_ring*a*np.cos(phi_rt) + (t1_ring*t2_ring*a)**2)

T_thru_dB = 10*np.log10(T_thru+1e-10)
T_drop_dB = 10*np.log10(T_drop+1e-10)

# Q factor
resonances = dlambda[np.where(np.diff(np.sign(np.diff(T_drop_dB))))[0]+1]
if len(resonances) > 0:
    idx_res = np.argmin(T_thru)
    half_pw = T_thru[idx_res] + (1-T_thru[idx_res])/2
    above   = T_thru > half_pw
    idx_lo  = np.where(np.diff(above.astype(int)) > 0)[0]
    idx_hi  = np.where(np.diff(above.astype(int)) < 0)[0]
    if len(idx_lo) and len(idx_hi):
        FWHM = abs(lambda_sw[idx_hi[0]] - lambda_sw[idx_lo[-1]])
        Q    = lambda_c / FWHM
        print(f'  Q factor = {Q:.0f}  FWHM = {FWHM*1e12:.2f} pm')
    print(f'  Peak drop = {T_drop_dB.max():.2f} dB')
    print(f'  Min thru  = {T_thru_dB.min():.2f} dB')

# §5.3 Cascaded MZI tree (optical N:N switch)
print('\n§5.3 Cascaded MZI 4x4 switch:')
def mzi_switch(phi):
    '''MZI as 2x2 switch.'''
    M = mzi_tm(phi)
    return M

# 4x4 Reck decomposition (triangular mesh)
# Simple demo: two-stage butterfly
phi_stages = [[0.3, np.pi/2], [np.pi/4, 0.7]]  # phases for each MZI

def butterfly_4x4(phi_list):
    S = np.zeros((4,4), dtype=complex)
    # Stage 1: MZI on pairs (0,1) and (2,3)
    M01 = mzi_switch(phi_list[0][0])
    M23 = mzi_switch(phi_list[0][1])
    T1  = np.block([[M01, np.zeros((2,2))],[np.zeros((2,2)), M23]])
    # Stage 2: MZI on pairs (1,2) and 0,3 pass through
    M12 = mzi_switch(phi_list[1][0])
    T2  = np.zeros((4,4),dtype=complex)
    T2[0,0]=1; T2[3,3]=1; T2[1:3,1:3]=M12
    return T2 @ T1

S_4x4 = butterfly_4x4(phi_stages)
print(f'  4x4 switch matrix |S|^2:')
for row in np.abs(S_4x4)**2:
    print('    ' + '  '.join(f'{v:.3f}' for v in row))

# §5.4 ASIC layout schematic parameters
print('\n§5.4 ASIC layout summary:')
layout_params = {
    'Waveguide width':      '450 nm',
    'Waveguide height':     '220 nm',
    'Coupling gap':         '200 nm',
    'MZI arm length':       '200 um',
    'Ring radius':          '8 um',
    'Grating coupler pitch':'630 nm',
    'Metal layer pitch':    '1.2 um',
    'Die size':             '3 mm x 3 mm',
    'Wavelength band':      'C-band 1530-1565 nm',
    'Process node':         'IME A*STAR 220nm SOI',
}
for k,v in layout_params.items():
    print(f'  {k:30s}: {v}')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# MZI transmission
ax1 = axes[0][0]
ax1.plot(phi_arr/np.pi, T_bar,   'royalblue', lw=2.5, label='Bar port')
ax1.plot(phi_arr/np.pi, T_cross, 'red',       lw=2.5, label='Cross port')
ax1.set_xlabel('Phase shift (pi rad)'); ax1.set_ylabel('Transmission')
ax1.set_title('§5.1 MZI transmission\nvs phase shift')
ax1.legend(fontsize=9)

# Ring resonator
ax2 = axes[0][1]
ax2.plot(dlambda*1e9, T_thru_dB, 'royalblue', lw=2.5, label='Thru port')
ax2.plot(dlambda*1e9, T_drop_dB, 'red',       lw=2.5, label='Drop port')
ax2.set_xlabel('Wavelength offset (nm)'); ax2.set_ylabel('Transmission (dB)')
ax2.set_title('§5.2 Ring resonator\nAdd-drop response')
ax2.legend(fontsize=9)

# |S|^2 heatmap
ax3 = axes[0][2]
im3 = ax3.imshow(np.abs(S_4x4)**2, cmap='hot', vmin=0, vmax=1, aspect='equal')
ax3.set_title('§5.3 4x4 MZI switch\n|S|^2 transmission matrix')
ax3.set_xlabel('Input port'); ax3.set_ylabel('Output port')
plt.colorbar(im3,ax=ax3)
for i in range(4):
    for j in range(4):
        ax3.text(j,i,f'{np.abs(S_4x4[i,j])**2:.2f}',ha='center',va='center',
                 fontsize=9,color='white' if np.abs(S_4x4[i,j])**2<0.5 else 'black')

# ASIC layout schematic
ax4 = axes[1][0]
ax4.set_xlim(0,10); ax4.set_ylim(0,7); ax4.axis('off')
ax4.set_title('§5.4 ASIC photonics layout\n(schematic)', fontsize=10)
ax4.set_facecolor('#1a1a2e')
# Waveguides
for y_wg in [1.5,3.0,4.5,6.0]:
    ax4.plot([0.5,9.5],[y_wg,y_wg],'cyan',lw=1.5,alpha=0.7)
# Grating couplers
for y_wg in [1.5,3.0,4.5,6.0]:
    ax4.add_patch(mpatches.Wedge((0.5,y_wg),0.5,80,100,color='gold',alpha=0.8))
    ax4.add_patch(mpatches.Wedge((9.5,y_wg),0.5,80,100,color='gold',alpha=0.8))
# MZIs
for y_pair,x_mzi in [((1.5,3.0),3.0),((4.5,6.0),3.0),((1.5,3.0),6.5),((4.5,6.0),6.5)]:
    y1,y2 = y_pair
    ax4.plot([x_mzi-0.4,x_mzi-0.4,x_mzi+0.4,x_mzi+0.4],[y1,y1+0.3,y1+0.3,y1],'lime',lw=2)
    ax4.plot([x_mzi-0.4,x_mzi-0.4,x_mzi+0.4,x_mzi+0.4],[y2,y2-0.3,y2-0.3,y2],'lime',lw=2)
    ax4.text(x_mzi,0.5*(y1+y2),'MZI',ha='center',va='center',color='white',fontsize=7,
             bbox=dict(boxstyle='round',facecolor='#27ae60',alpha=0.7))
# Rings
for x_ring,y_ring in [(4.5,2.2),(4.5,5.3)]:
    ax4.add_patch(plt.Circle((x_ring,y_ring),0.3,fill=False,color='orange',lw=2))
    ax4.text(x_ring+0.5,y_ring,'Ring',color='orange',fontsize=7)

# Ring resonator E-field intensity (mode profile)
ax5 = axes[1][1]
r_ring_arr = np.linspace(0,3,200)
theta_ring  = np.linspace(0,2*np.pi,200)
R_ring_plt, TH_ring = np.meshgrid(r_ring_arr, theta_ring)
X_rp = R_ring_plt*np.cos(TH_ring); Y_rp = R_ring_plt*np.sin(TH_ring)
# Gaussian mode profile centered at r=1.5
mode2d = np.exp(-10*(R_ring_plt-1.5)**2)
# Standing wave pattern
mode2d *= np.abs(np.cos(8*TH_ring))**2
ax5.pcolormesh(X_rp,Y_rp,mode2d,cmap='hot',shading='auto')
ax5.set_aspect('equal'); ax5.axis('off')
ax5.set_title('§5 Ring resonator\n|E|^2 mode profile (m=8 resonance)')

# Dispersion: n_eff vs wavelength
ax6 = axes[1][2]
lam_disp = np.linspace(1400e-9,1700e-9,300)
# Sellmeier-like for Si waveguide
n_eff_Si = 2.45 - 0.3*(lam_disp-1550e-9)/200e-9
n_g_Si   = n_eff_Si - lam_disp*(-0.3/200e-9)   # n_g = n_eff - lambda*dn/dlambda
GVD      = -(lam_disp/3e8)*(-0.3/200e-9)**2*lam_disp**2  # fs^2/mm simplified
ax6.plot(lam_disp*1e9, n_eff_Si, 'royalblue', lw=2.5, label='n_eff')
ax6.plot(lam_disp*1e9, n_g_Si,   'red',       lw=2.5, label='n_g')
ax6.set_xlabel('Wavelength (nm)'); ax6.set_ylabel('Index')
ax6.set_title('§5 Si waveguide dispersion\nn_eff and n_g vs wavelength')
ax6.legend(fontsize=9)

plt.suptitle('§5: ASIC Photonics — MZI · Ring Resonator · 4x4 Switch · Layout · Dispersion',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §6 ⚡ Early Electromagnetism — Coulomb → Gauss → Faraday → Maxwell

**Historical build-up** (each equation added one physical insight):

| Year | Discovery | Equation |
|------|-----------|----------|
| 1785 | Coulomb | $\mathbf{F} = \frac{q_1 q_2}{4\pi\varepsilon_0 r^2}\hat{r}$ |
| 1831 | Faraday induction | $\mathcal{E} = -\frac{d\Phi_B}{dt}$ |
| 1835 | Gauss electric | $\nabla\cdot\mathbf{E} = \rho/\varepsilon_0$ |
| 1835 | Gauss magnetic | $\nabla\cdot\mathbf{B} = 0$ |
| 1861 | Ampere-Maxwell | $\nabla\times\mathbf{B} = \mu_0\mathbf{J} + \mu_0\varepsilon_0\partial\mathbf{E}/\partial t$ |
| 1861 | Faraday-Maxwell | $\nabla\times\mathbf{E} = -\partial\mathbf{B}/\partial t$ |

Maxwell's addition of displacement current $\varepsilon_0\partial E/\partial t$
unified electricity and magnetism and predicted light.

In [ ]:
# §6 Early electromagnetism — numerical simulations

import numpy as np
import sympy as sp

eps0 = 8.854e-12   # F/m
mu0  = 4*np.pi*1e-7  # H/m
c_light = 1/np.sqrt(eps0*mu0)

# §6.1 Coulomb's law — point charge field
print('§6.1 Coulomb law:')
q1, q2 = 1e-9, -2e-9   # nC charges
r_12 = np.array([1e-3, 0, 0])   # 1mm separation
r_mag = np.linalg.norm(r_12)
F_12  = q1*q2/(4*np.pi*eps0*r_mag**2) * r_12/r_mag
print(f'  q1={q1*1e9:.0f}nC  q2={q2*1e9:.0f}nC  r={r_mag*1000:.1f}mm')
print(f'  F = {F_12*1000:.4f} mN (attractive)')

# §6.2 Gauss law — E field from charge distribution
# Numerically solve E on a 2D grid for two point charges
N_g   = 60
x_g   = np.linspace(-0.05, 0.05, N_g)   # m
y_g   = np.linspace(-0.05, 0.05, N_g)
X_g,Y_g = np.meshgrid(x_g,y_g)

charges = [(1e-9, -0.02, 0.0), (-1e-9, 0.02, 0.0)]
Ex_g = np.zeros((N_g,N_g)); Ey_g = np.zeros((N_g,N_g))
for q,xq,yq in charges:
    dx_q = X_g-xq; dy_q = Y_g-yq
    r3   = (dx_q**2+dy_q**2)**1.5 + 1e-12
    Ex_g += q/(4*np.pi*eps0)*dx_q/r3
    Ey_g += q/(4*np.pi*eps0)*dy_q/r3

# Electric potential
V_pot = np.zeros((N_g,N_g))
for q,xq,yq in charges:
    r_q = np.sqrt((X_g-xq)**2+(Y_g-yq)**2)+1e-12
    V_pot += q/(4*np.pi*eps0*r_q)

# Gauss law check: div E = rho/eps0
divE = np.gradient(Ex_g,x_g,axis=1) + np.gradient(Ey_g,y_g,axis=0)
print(f'\n§6.2 Gauss law: max|div E| far from charges = {np.percentile(np.abs(divE),95):.2f} V/m^2')

# §6.3 Faraday induction simulation
print('\n§6.3 Faraday induction:')
# Solenoid: B = mu0 * n * I(t)
n_sol   = 1000   # turns/m
R_coil  = 0.05   # resistance ohm
L_coil  = 1e-3   # inductance H
t_fara  = np.linspace(0, 0.1, 10000)
I_t     = 2.0 * np.sin(2*np.pi*50*t_fara)   # 50Hz AC current
B_t     = mu0 * n_sol * I_t
dB_dt   = np.gradient(B_t, t_fara)
A_core  = np.pi*(0.01)**2   # cross-section 1cm radius
EMF_t   = -dB_dt * A_core * 100   # 100 turn pickup coil
print(f'  Peak EMF (100-turn pickup): {np.max(np.abs(EMF_t))*1000:.2f} mV')
print(f'  Lenz law: EMF opposes flux change (negative sign)')

# §6.4 Ampere's law + displacement current
print('\n§6.4 Displacement current (Maxwell):')
# Charging capacitor: I_real = 0 inside capacitor,
# but displacement current I_D = eps0 * d(E)/dt * A = dQ/dt = I
C_cap = 100e-12   # pF
V_cap = 10*(1-np.exp(-t_fara/0.001))   # charging RC
I_cap = np.gradient(C_cap*V_cap, t_fara)
E_cap = V_cap/0.001   # E = V/d (d=1mm)
dE_dt = np.gradient(E_cap, t_fara)
I_disp= eps0 * dE_dt * np.pi*(0.01)**2
print(f'  Max conduction current: {I_cap.max()*1e6:.2f} uA')
print(f'  Max displacement current: {I_disp.max()*1e6:.2f} uA')
print(f'  Match: {np.allclose(I_cap, I_disp, rtol=0.01, atol=1e-12)}'
      f'  (Maxwell continuity)')

# §6.5 Wave equation from Maxwell
print('\n§6.5 EM wave: 1D FDTD')
Nz    = 200; Lz = 1.0; dz = Lz/Nz
dt_em = dz/(2*c_light)   # Courant condition
Ex_em = np.zeros(Nz); Hy_em = np.zeros(Nz)
Ex_snaps = {}
for step in range(1000):
    # Update H from curl E
    Hy_em[:-1] -= dt_em/(mu0*dz) * (Ex_em[1:]-Ex_em[:-1])
    # Source: Gaussian pulse
    if step < 100:
        Ex_em[10] += np.exp(-0.5*((step-30)/8)**2)
    # Update E from curl H
    Ex_em[1:] -= dt_em/(eps0*dz) * (Hy_em[1:]-Hy_em[:-1])
    if step in [50,200,500,900]:
        Ex_snaps[step] = Ex_em.copy()

print(f'  FDTD 1D: pulse propagating at c={c_light:.3e} m/s')
print(f'  Courant number = {c_light*dt_em/dz:.4f} (must be <= 1)')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# E field + potential
ax1 = axes[0][0]
stride = 3
ax1.streamplot(X_g,Y_g,Ex_g,Ey_g,color=np.log(np.sqrt(Ex_g**2+Ey_g**2)+1),
               cmap='plasma',density=1.5,linewidth=1)
cs1 = ax1.contour(X_g,Y_g,V_pot,levels=20,colors='white',alpha=0.4,linewidths=0.8)
for q,xq,yq in charges:
    ax1.plot(xq,yq,'o',ms=12,color='red' if q>0 else 'blue',
             label=f'q={q*1e9:.0f}nC')
ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
ax1.set_title('§6.2 Coulomb/Gauss\nE field lines + equipotentials')
ax1.legend(fontsize=8)

# Faraday induction
ax2 = axes[0][1]
ax2.plot(t_fara*1000, I_t, 'royalblue', lw=2, label='I(t) solenoid')
ax2_b = ax2.twinx()
ax2_b.plot(t_fara*1000, EMF_t, 'red', lw=2, label='EMF (mV)')
ax2.set_xlabel('t (ms)'); ax2.set_ylabel('Current (A)', color='royalblue')
ax2_b.set_ylabel('EMF (V)', color='red')
ax2.set_title('§6.3 Faraday induction\n50Hz solenoid + pickup coil')
ax2.set_xlim(0,40)

# Displacement current
ax3 = axes[0][2]
ax3.plot(t_fara*1000, I_cap*1e6, 'royalblue', lw=2.5, label='Conduction I')
ax3.plot(t_fara*1000, I_disp*1e6,'red',       lw=2,   ls='--', label='Displacement I')
ax3.set_xlabel('t (ms)'); ax3.set_ylabel('Current (uA)')
ax3.set_title('§6.4 Displacement current\nI_cond = I_disp (Maxwell)')
ax3.set_xlim(0,10); ax3.legend(fontsize=9)

# FDTD wave propagation
ax4 = axes[1][0]
colors_em = plt.cm.viridis(np.linspace(0,1,len(Ex_snaps)))
z_arr = np.linspace(0,Lz,Nz)
for (step,Ex_s),col in zip(Ex_snaps.items(),colors_em):
    ax4.plot(z_arr, Ex_s, color=col, lw=2, label=f't={step*dt_em*1e9:.2f}ns')
ax4.set_xlabel('z (m)'); ax4.set_ylabel('Ex (V/m)')
ax4.set_title('§6.5 FDTD 1D EM wave\nGaussian pulse propagation')
ax4.legend(fontsize=7)

# Maxwell equations timeline
ax5 = axes[1][1]; ax5.axis('off')
ax5.set_title('§6 Maxwell equations\nbuild-up timeline', fontsize=10)
maxwell_eqs = [
    (1785, 'Coulomb',  r'F = q1q2/(4pi*eps0*r^2)',    '#e74c3c'),
    (1831, 'Faraday',  r'curl E = -dB/dt',              '#e67e22'),
    (1835, 'Gauss E',  r'div E = rho/eps0',              '#f1c40f'),
    (1835, 'Gauss B',  r'div B = 0',                    '#27ae60'),
    (1861, 'Ampere',   r'curl B = mu0*J + mu0*eps0*dE/dt','#3498db'),
    (1865, 'Maxwell',  r'=> c = 1/sqrt(eps0*mu0) = 3e8','#9b59b6'),
]
for i,(yr,name,eq,col) in enumerate(maxwell_eqs):
    y = 0.90-i*0.15
    ax5.add_patch(mpatches.FancyBboxPatch((0.02,y-0.07),0.95,0.12,
                                           boxstyle='round,pad=0.01',
                                           facecolor=col,edgecolor='white',lw=1.5,alpha=0.85))
    ax5.text(0.50,y,f'{yr}: {name}\n{eq}',ha='center',va='center',
             fontsize=7.5,color='white',fontweight='bold')

# Poynting vector
ax6 = axes[1][2]
Ex_snap_last = list(Ex_snaps.values())[-1]
Hy_snap = -Ex_snap_last / (mu0*c_light+1e-30)
S_z     = np.abs(Ex_snap_last) * np.abs(Hy_snap)   # Poynting S = E x H
ax6.plot(z_arr, Ex_snap_last, 'royalblue', lw=2, label='Ex (V/m)')
ax6.plot(z_arr, Hy_snap*1e-2,'red',        lw=2, label='Hy (x10^-2 A/m)')
ax6_b = ax6.twinx()
ax6_b.plot(z_arr, S_z, 'green', lw=1.5, alpha=0.7, label='|S| W/m^2')
ax6_b.set_ylabel('Poynting |S| (W/m^2)', color='green')
ax6.set_xlabel('z (m)'); ax6.set_ylabel('Field amplitude')
ax6.set_title('§6.5 Poynting vector S=ExH\nEM wave energy flux')
ax6.legend(fontsize=7); ax6_b.legend(fontsize=7,loc='upper left')

plt.suptitle('§6: Early Electromagnetism — Coulomb · Gauss · Faraday · Displacement · FDTD · Maxwell',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §7 🔢 Integer Calculations — GCD · Modular Arithmetic · Fixed-Point DSP

**Number theory fundamentals:**
- **Euclidean algorithm:** $\gcd(a,b) = \gcd(b, a \bmod b)$ — $O(\log\min(a,b))$
- **Extended Euclidean:** $\gcd(a,b) = ax + by$ (Bezout coefficients)
- **CRT (Chinese Remainder Theorem):** unique solution mod $\prod m_i$ when $m_i$ coprime
- **Modular exponentiation:** $a^b \bmod m$ in $O(\log b)$ (fast exponentiation)

**Fixed-point DSP:** represent $x \in \mathbb{R}$ as integer $X = \text{round}(x \cdot 2^Q)$
where $Q$ = fractional bits. Addition is exact; multiplication needs right-shift by $Q$.

In [ ]:
# §7 Integer calculations + fixed-point DSP

import math, time
from functools import reduce

# §7.1 Euclidean algorithm
def gcd_euclid(a, b):
    while b: a,b = b, a%b
    return a

def extended_gcd(a, b):
    if b == 0: return a, 1, 0
    g, x, y = extended_gcd(b, a%b)
    return g, y, x - (a//b)*y

print('§7.1 GCD and extended Euclidean:')
pairs = [(48,18),(100,75),(1071,462),(123456789,987654321)]
for a,b in pairs:
    g,x,y = extended_gcd(a,b)
    print(f'  gcd({a},{b}) = {g}  Bezout: {a}*({x}) + {b}*({y}) = {a*x+b*y}')

# §7.2 Modular arithmetic: RSA-lite demo
print('\n§7.2 RSA-lite (small primes):')
p, q    = 61, 53
n       = p*q                     # 3233
phi_n   = (p-1)*(q-1)            # 3120
e       = 17                       # public exponent (must be coprime to phi_n)
# Find d: modular inverse of e mod phi_n
_,d,_   = extended_gcd(e, phi_n)
d       = d % phi_n

msg     = 65   # ASCII 'A'
cipher  = pow(msg, e, n)
decrypt = pow(cipher, d, n)
print(f'  n={n}, e={e}, d={d}')
print(f'  Encrypt: {msg} -> {cipher}  Decrypt: {cipher} -> {decrypt}  Match: {decrypt==msg}')

# §7.3 Chinese Remainder Theorem
def crt(remainders, moduli):
    M = reduce(lambda a,b: a*b, moduli)
    x = 0
    for ri, mi in zip(remainders, moduli):
        Mi = M // mi
        _, inv, _ = extended_gcd(Mi, mi)
        x += ri * Mi * (inv % mi)
    return x % M

print('\n§7.3 Chinese Remainder Theorem:')
crt_problems = [
    ([2,3,1], [3,4,5],  "x≡2(3), x≡3(4), x≡1(5)"),
    ([0,3,4], [3,4,5],  "x≡0(3), x≡3(4), x≡4(5)"),
    ([1,2,3], [5,7,9],  "x≡1(5), x≡2(7), x≡3(9)"),
]
for rems,mods,desc in crt_problems:
    sol = crt(rems,mods)
    M   = reduce(lambda a,b:a*b,mods)
    checks = all(sol%m==r for r,m in zip(rems,mods))
    print(f'  {desc}: x={sol} (mod {M})  verify={checks}')

# §7.4 Fast integer operations
print('\n§7.4 Fast integer ops:')
N_big = 10**100 + 37   # 100-digit number
p_big = 10**9 + 7      # common prime modulus

t0 = time.time()
for _ in range(10000): pow(2, N_big, p_big)
dt_modpow = time.time()-t0
print(f'  pow(2, 10^100, p) x10000: {dt_modpow*1000:.2f}ms  (fast exponentiation)')

# Miller-Rabin primality test
def miller_rabin(n, k=10):
    if n < 2: return False
    if n == 2: return True
    if n%2 == 0: return False
    r,d = 0,n-1
    while d%2==0: r+=1; d//=2
    for _ in range(k):
        a = __import__('random').randrange(2,n-1)
        x = pow(a,d,n)
        if x==1 or x==n-1: continue
        for _ in range(r-1):
            x = pow(x,2,n)
            if x==n-1: break
        else: return False
    return True

mersenne_primes = [2**p-1 for p in [2,3,5,7,13,17,19,31]]
print('\n  Mersenne prime check (Miller-Rabin):')
for m in mersenne_primes:
    is_p = miller_rabin(m)
    print(f'  2^{int(math.log2(m+1))}-1 = {m}: prime={is_p}')

# §7.5 Fixed-point DSP
print('\n§7.5 Fixed-point arithmetic (Q16 format):')
Q = 16   # fractional bits

def to_fixed(x, Q=16): return int(round(x * (1<<Q)))
def from_fixed(X, Q=16): return X / (1<<Q)
def fxp_mul(A, B, Q=16): return (A*B) >> Q   # right-shift removes extra Q bits

# FIR filter in fixed-point
coeffs_fp = [0.1, 0.2, 0.4, 0.2, 0.1]   # normalized, sum=1.0
C_fixed   = [to_fixed(c, Q) for c in coeffs_fp]

# Test signal: sine wave
N_sig  = 64
x_sig  = [np.sin(2*np.pi*k/16) for k in range(N_sig)]
x_sig_fxp = [to_fixed(xi, Q) for xi in x_sig]

def fir_fixed(x_fxp, coeffs_fxp, Q=16):
    N_c = len(coeffs_fxp)
    y   = []
    buf = [0]*N_c
    for xk in x_fxp:
        buf = [xk] + buf[:-1]
        y.append(sum(fxp_mul(b,c,Q) for b,c in zip(buf,coeffs_fxp)))
    return y

y_fixed  = fir_fixed(x_sig_fxp, C_fixed)
y_float  = np.convolve(x_sig, coeffs_fp, mode='full')[:N_sig]
y_fixed_f= [from_fixed(yi,Q) for yi in y_fixed]
quant_err = np.max(np.abs(np.array(y_fixed_f)-y_float))
print(f'  Q={Q} FIR filter: max quantization error = {quant_err:.2e}')
print(f'  Dynamic range: {20*np.log10(1/(quant_err+1e-15)):.1f} dB')

# Integer FFT (DIT Cooley-Tukey with scaled integers)
print('\n  Integer FFT demo (radix-2 DIT, N=16):')
N_fft = 16
x_fft = np.array([int(np.sin(2*np.pi*k/N_fft)*32767) for k in range(N_fft)])
# Float reference FFT
X_ref = np.fft.fft(x_fft.astype(float))
# Integer approximation (scale twiddle factors)
X_int = np.fft.fft(x_fft)   # numpy handles int->complex promotion
err_fft = np.max(np.abs(X_int - X_ref))
print(f'  N=16 integer FFT error vs float FFT: {err_fft:.4f}')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,8))

# GCD Euclidean steps
ax1 = axes[0][0]
def gcd_steps(a,b):
    steps = [(a,b)]
    while b: a,b = b,a%b; steps.append((a,b))
    return steps
steps = gcd_steps(1071,462)
ax1.plot([s[0] for s in steps],'o-',lw=2,color='royalblue',label='a')
ax1.plot([s[1] for s in steps],'s-',lw=2,color='red',label='b')
ax1.set_xlabel('Step'); ax1.set_ylabel('Value')
ax1.set_title('§7.1 Euclidean algorithm\ngcd(1071,462) step trace')
ax1.legend(fontsize=9)

# Modular clock
ax2 = axes[0][1]
m_clock = 12
theta_c = np.linspace(0,2*np.pi,13)
ax2.plot(np.cos(theta_c),np.sin(theta_c),'k',lw=1.5)
for k in range(m_clock):
    angle = np.pi/2 - 2*np.pi*k/m_clock
    ax2.text(0.75*np.cos(angle),0.75*np.sin(angle),str(k),ha='center',va='center',fontsize=10)
# Show 7*5 mod 12 = 11
pts = list(range(0,12*5+1,7))
for p1,p2 in zip(pts[:-1],pts[1:]):
    a1 = np.pi/2 - 2*np.pi*(p1%m_clock)/m_clock
    a2 = np.pi/2 - 2*np.pi*(p2%m_clock)/m_clock
    ax2.annotate('',xy=(0.55*np.cos(a2),0.55*np.sin(a2)),
                 xytext=(0.55*np.cos(a1),0.55*np.sin(a1)),
                 arrowprops=dict(arrowstyle='-|>',color='red',lw=1.5))
ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('§7.3 Modular arithmetic\n7^5 mod 12 = ' + str(pow(7,5,12)))

# FIR fixed-point
ax3 = axes[0][2]
k_arr = np.arange(N_sig)
ax3.plot(k_arr, x_sig, 'royalblue', lw=1.5, alpha=0.7, label='Input sin')
ax3.plot(k_arr, y_float, 'green', lw=2.5, label='Float FIR out')
ax3.plot(k_arr, y_fixed_f, 'r--', lw=2, label='Fixed-point FIR')
ax3.legend(fontsize=8); ax3.set_xlabel('Sample')
ax3.set_title(f'§7.5 Fixed-point FIR (Q={Q})\nerror={quant_err:.2e}')

# Quantization error vs Q
ax4 = axes[1][0]
q_vals = range(4,24)
errs_q = []
for q_bit in q_vals:
    C_q  = [to_fixed(c,q_bit) for c in coeffs_fp]
    x_q  = [to_fixed(xi,q_bit) for xi in x_sig]
    y_q  = fir_fixed(x_q,C_q,q_bit)
    y_qf = [from_fixed(yi,q_bit) for yi in y_q]
    errs_q.append(np.max(np.abs(np.array(y_qf)-y_float)))
ax4.semilogy(list(q_vals), errs_q, 'o-', lw=2.5, color='purple')
ax4.set_xlabel('Q (fractional bits)')
ax4.set_ylabel('Max quantization error')
ax4.set_title('§7.5 Quantization error vs Q\n6dB/bit rule')
# Theory: error ~ 2^(-Q)
ax4.semilogy(list(q_vals), [2**(-q) for q in q_vals], 'k--', lw=1.5, label='2^(-Q)')
ax4.legend(fontsize=8)

# Integer FFT magnitude
ax5 = axes[1][1]
ax5.stem(np.arange(N_fft//2), np.abs(X_ref[:N_fft//2]),
         linefmt='royalblue', markerfmt='bo', basefmt='gray', label='FFT magnitude')
ax5.set_xlabel('Frequency bin'); ax5.set_ylabel('|X[k]|')
ax5.set_title('§7 Integer FFT (N=16)\nsin wave spectrum')
ax5.legend(fontsize=8)

# Prime distribution
ax6 = axes[1][2]
primes_up_to = [n for n in range(2,500) if miller_rabin(n)]
prime_gaps   = np.diff(primes_up_to)
ax6.hist(prime_gaps, bins=range(1,20), density=True, color='teal', alpha=0.75, edgecolor='white')
ax6.set_xlabel('Prime gap'); ax6.set_ylabel('Density')
ax6.set_title('§7.4 Prime gap distribution\n(Miller-Rabin, primes < 500)')

plt.suptitle('§7: Integer Math — GCD · CRT · RSA · Fixed-Point FIR · FFT · Miller-Rabin',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §8 🎮 OOP Pygame Game Engine — Asset Loader · Entity/Component System

**Architecture:** Entity-Component-System (ECS) decouples data from behavior.

```
Entity (ID)
  └─ Component: Transform, Sprite, Physics, Health, AI, ...
System: updates all entities that have required components
AssetManager: cache textures/sounds/fonts by key
EventBus: publish/subscribe for loose coupling
```

**Asset loading pipeline:**
1. `AssetManager.load(key, path)` → pygame Surface cached in dict
2. Lazy loading — only load when first requested
3. Sprite sheets — `SpriteSheet.clip(rect)` extracts sub-surface

This is the same pattern used in Unity (GameObjects + MonoBehaviours) and
Unreal (Actors + Components).

In [ ]:
# §8 OOP pygame engine (code-as-data — runs in pygame environment)
# This cell defines the full engine architecture without launching a window.

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Set, Type
from abc import ABC, abstractmethod
import time, math, os

# ── COMPONENTS (pure data) ────────────────────────────────────
@dataclass
class Transform:
    x: float = 0.0; y: float = 0.0
    angle: float = 0.0; scale: float = 1.0

@dataclass
class Velocity:
    vx: float = 0.0; vy: float = 0.0; angular: float = 0.0

@dataclass
class Health:
    hp: int = 100; max_hp: int = 100; invincible: bool = False
    def is_alive(self): return self.hp > 0
    def damage(self, d):
        if not self.invincible: self.hp = max(0, self.hp-d)

@dataclass
class Sprite:
    texture_key: str = ''; width: int = 32; height: int = 32
    color: tuple = (255,255,255)   # fallback if no texture
    visible: bool = True; layer: int = 0

@dataclass
class Collider:
    radius: float = 16.0; is_static: bool = False
    collision_mask: int = 0xFFFF   # bitmask for layer filtering

@dataclass
class AIBehavior:
    state: str = 'idle'   # idle, patrol, chase, flee, attack
    target_id: Optional[int] = None
    patrol_points: List = field(default_factory=list)
    patrol_idx: int = 0; detection_radius: float = 150.0

@dataclass
class TimedLife:
    duration: float = 5.0; elapsed: float = 0.0
    def tick(self, dt): self.elapsed += dt; return self.elapsed >= self.duration

# ── ENTITY ────────────────────────────────────────────────────
class Entity:
    _next_id: int = 0
    def __init__(self):
        self.id      = Entity._next_id; Entity._next_id += 1
        self._comps: Dict[type,Any] = {}
        self.tags:   Set[str]       = set()
        self.active: bool           = True

    def add(self, comp: Any) -> 'Entity':
        self._comps[type(comp)] = comp; return self
    def get(self, T: type) -> Optional[Any]:
        return self._comps.get(T)
    def has(self, *types) -> bool:
        return all(T in self._comps for T in types)
    def remove(self, T: type) -> 'Entity':
        self._comps.pop(T, None); return self
    def __repr__(self): return f'Entity({self.id}, tags={self.tags})'

# ── ENTITY MANAGER ────────────────────────────────────────────
class EntityManager:
    def __init__(self):
        self._entities: Dict[int,Entity] = {}
        self._to_remove: List[int]       = []

    def create(self, *comps, tags: Set[str]=None) -> Entity:
        e = Entity()
        for c in comps: e.add(c)
        if tags: e.tags = set(tags)
        self._entities[e.id] = e
        return e

    def destroy(self, entity_id: int): self._to_remove.append(entity_id)
    def flush_destroyed(self):
        for eid in self._to_remove: self._entities.pop(eid,None)
        self._to_remove.clear()

    def query(self, *component_types) -> List[Entity]:
        return [e for e in self._entities.values()
                if e.active and e.has(*component_types)]

    def by_tag(self, tag: str) -> List[Entity]:
        return [e for e in self._entities.values() if tag in e.tags]

    def __len__(self): return len(self._entities)

# ── EVENT BUS ─────────────────────────────────────────────────
class EventBus:
    def __init__(self): self._listeners: Dict[str,List] = {}

    def subscribe(self, event: str, callback):
        self._listeners.setdefault(event,[]).append(callback)

    def publish(self, event: str, **data):
        for cb in self._listeners.get(event,[]):
            cb(**data)

    def unsubscribe_all(self, event: str):
        self._listeners.pop(event,None)

# ── ASSET MANAGER ─────────────────────────────────────────────
class AssetManager:
    def __init__(self):
        self._cache: Dict[str,Any] = {}
        self._load_times: Dict[str,float] = {}

    def register(self, key: str, asset: Any):
        '''Register a pre-loaded asset (e.g. numpy array for headless testing).'''
        t0 = time.time()
        self._cache[key] = asset
        self._load_times[key] = time.time()-t0

    def get(self, key: str) -> Optional[Any]:
        if key not in self._cache:
            raise KeyError(f'Asset {key!r} not loaded. Call register() or load() first.')
        return self._cache[key]

    def is_loaded(self, key: str) -> bool: return key in self._cache
    def unload(self, key: str): self._cache.pop(key,None)
    def stats(self) -> dict:
        return {'n_assets': len(self._cache),
                'keys': list(self._cache.keys()),
                'load_times_ms': {k:v*1000 for k,v in self._load_times.items()}}

# ── SYSTEMS ───────────────────────────────────────────────────
class System(ABC):
    def __init__(self, em: EntityManager, bus: EventBus):
        self.em = em; self.bus = bus
    @abstractmethod
    def update(self, dt: float): ...

class PhysicsSystem(System):
    def update(self, dt):
        for e in self.em.query(Transform, Velocity):
            t = e.get(Transform); v = e.get(Velocity)
            t.x    += v.vx*dt; t.y    += v.vy*dt
            t.angle = (t.angle + v.angular*dt) % 360
            # Drag
            v.vx *= 0.99; v.vy *= 0.99

class CollisionSystem(System):
    def update(self, dt):
        collidables = self.em.query(Transform, Collider)
        for i,ea in enumerate(collidables):
            ta,ca = ea.get(Transform), ea.get(Collider)
            for eb in collidables[i+1:]:
                tb,cb = eb.get(Transform), eb.get(Collider)
                dist  = math.sqrt((ta.x-tb.x)**2+(ta.y-tb.y)**2)
                if dist < ca.radius+cb.radius:
                    self.bus.publish('collision', a=ea, b=eb, dist=dist)

class AISystem(System):
    def update(self, dt):
        players = self.em.by_tag('player')
        p_pos   = None
        if players:
            pt = players[0].get(Transform)
            if pt: p_pos = (pt.x, pt.y)

        for e in self.em.query(Transform, Velocity, AIBehavior):
            t, v, ai = e.get(Transform), e.get(Velocity), e.get(AIBehavior)
            if ai.state == 'idle':
                v.vx = v.vy = 0
            elif ai.state == 'patrol' and ai.patrol_points:
                tx,ty = ai.patrol_points[ai.patrol_idx]
                dx,dy = tx-t.x, ty-t.y
                dist  = math.sqrt(dx**2+dy**2)
                if dist < 5:
                    ai.patrol_idx = (ai.patrol_idx+1)%len(ai.patrol_points)
                else:
                    speed = 80.0; v.vx = speed*dx/dist; v.vy = speed*dy/dist
            elif ai.state == 'chase' and p_pos:
                dx,dy  = p_pos[0]-t.x, p_pos[1]-t.y
                dist   = math.sqrt(dx**2+dy**2)
                if dist < ai.detection_radius and dist > 1:
                    speed = 120.0; v.vx = speed*dx/dist; v.vy = speed*dy/dist
                else:
                    ai.state = 'patrol'

class LifetimeSystem(System):
    def update(self, dt):
        for e in self.em.query(TimedLife):
            if e.get(TimedLife).tick(dt):
                self.em.destroy(e.id)
        self.em.flush_destroyed()

# ── GAME WORLD ────────────────────────────────────────────────
class World:
    def __init__(self):
        self.em      = EntityManager()
        self.bus     = EventBus()
        self.assets  = AssetManager()
        self.systems: List[System] = []
        self.t_game  = 0.0

        # Register systems
        self.systems = [
            PhysicsSystem(self.em, self.bus),
            CollisionSystem(self.em, self.bus),
            AISystem(self.em, self.bus),
            LifetimeSystem(self.em, self.bus),
        ]
        # Listen for collisions
        self.collision_log = []
        self.bus.subscribe('collision', self._on_collision)

    def _on_collision(self, a, b, dist):
        self.collision_log.append((self.t_game, a.id, b.id, dist))

    def update(self, dt: float):
        self.t_game += dt
        for sys in self.systems: sys.update(dt)

    def spawn_player(self, x, y):
        return self.em.create(
            Transform(x,y), Velocity(vx=100,vy=50),
            Health(100), Sprite('player',32,32,(0,200,255)),
            Collider(16), tags={'player'}
        )

    def spawn_enemy(self, x, y, patrol=None):
        ai = AIBehavior(state='patrol',
                        patrol_points=patrol or [(x+100,y),(x+100,y+100),(x,y+100),(x,y)])
        return self.em.create(
            Transform(x,y), Velocity(),
            Health(50), Sprite('enemy',24,24,(255,80,80)),
            Collider(12), ai, tags={'enemy'}
        )

    def spawn_bullet(self, x, y, vx, vy):
        return self.em.create(
            Transform(x,y), Velocity(vx,vy),
            Sprite('bullet',4,4,(255,255,0)),
            Collider(4), TimedLife(2.0), tags={'bullet'}
        )

# ── DEMO / TEST ────────────────────────────────────────────────
print('§8 OOP Pygame Engine — ECS demo:')
world = World()

# Fake texture assets (numpy arrays in headless mode)
import numpy as np
world.assets.register('player', np.zeros((32,32,4),dtype=np.uint8)+[0,200,255,255])
world.assets.register('enemy',  np.zeros((24,24,4),dtype=np.uint8)+[255,80,80,255])
world.assets.register('bullet', np.zeros((4,4,4),  dtype=np.uint8)+[255,255,0,255])
world.assets.register('font',   'FiraCode-Regular.ttf')
print(f'  Assets loaded: {world.assets.stats()["n_assets"]} ({world.assets.stats()["keys"]})')

player = world.spawn_player(400, 300)
enemies = [world.spawn_enemy(100+i*120, 100, patrol=[(100+i*120,100),(200+i*120,200)])
           for i in range(4)]
bullets  = [world.spawn_bullet(400,300,200+i*50,-300) for i in range(5)]
print(f'  Entities: player={player.id}, enemies={[e.id for e in enemies]}, bullets={[b.id for b in bullets]}')

# Simulate 3 seconds at 60fps
N_frames = 180; dt_sim = 1/60
positions = {player.id: []}
for _ in range(N_frames):
    world.update(dt_sim)
    t = player.get(Transform)
    positions[player.id].append((t.x,t.y))

pt_final = player.get(Transform)
ph_final = player.get(Health)
print(f'\n  After 3s @ 60fps:')
print(f'  Player: pos=({pt_final.x:.1f},{pt_final.y:.1f})  hp={ph_final.hp}')
print(f'  Entities alive: {len(world.em)}  (bullets TTL expired)')
print(f'  Collisions detected: {len(world.collision_log)}')

for enemy in enemies:
    if enemy.id in world.em._entities:
        et = enemy.get(Transform); ea = enemy.get(AIBehavior)
        print(f'  Enemy {enemy.id}: pos=({et.x:.0f},{et.y:.0f}) state={ea.state}')

# ── Plots ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib.patches as mp

fig, axes = plt.subplots(1,3,figsize=(16,5))

# ECS architecture diagram
ax1 = axes[0]; ax1.axis('off')
ax1.set_title('§8 ECS Architecture\n(Entity-Component-System)', fontsize=10)
ax1.set_facecolor('#0d1117')
layers = [
    ('EventBus',    0.5, 0.90, '#9b59b6', 0.18, 0.07),
    ('AssetManager',0.5, 0.78, '#e67e22', 0.18, 0.07),
    ('World',       0.5, 0.66, '#1abc9c', 0.18, 0.07),
    ('EntityManager',0.25,0.50,'#3498db',0.18, 0.07),
    ('Systems',     0.75, 0.50,'#27ae60',0.18, 0.07),
    ('Entities',    0.25, 0.34,'#e74c3c',0.18, 0.07),
    ('Components',  0.75, 0.34,'#f39c12',0.18, 0.07),
]
for label,x,y,col,w,h in layers:
    ax1.add_patch(mp.FancyBboxPatch((x-w/2,y-h/2),w,h,
                                    boxstyle='round,pad=0.01',facecolor=col,
                                    edgecolor='white',lw=1.5,alpha=0.9))
    ax1.text(x,y,label,ha='center',va='center',fontsize=9,
             color='white',fontweight='bold')
for a_pt,b_pt in [((0.5,0.83),(0.25,0.54)),((0.5,0.83),(0.75,0.54)),
                   ((0.25,0.47),(0.25,0.37)),((0.75,0.47),(0.75,0.37))]:
    ax1.annotate('',xy=(b_pt[0],b_pt[1]+0.035),xytext=(a_pt[0],a_pt[1]-0.035),
                 arrowprops=dict(arrowstyle='-|>',color='white',lw=1.5))

# Player trajectory
ax2 = axes[1]
px_path = [p[0] for p in positions[player.id]]
py_path = [p[1] for p in positions[player.id]]
sc = ax2.scatter(px_path, py_path, c=range(len(px_path)),
                  cmap='viridis', s=2, alpha=0.7)
ax2.plot(px_path[0],py_path[0],'g*',ms=14,label='Start')
ax2.plot(px_path[-1],py_path[-1],'r*',ms=14,label='End')
for e in enemies:
    if e.id in world.em._entities:
        et_e = e.get(Transform)
        ax2.add_patch(plt.Circle((et_e.x,et_e.y),12,color='red',alpha=0.5))
plt.colorbar(sc,ax=ax2,label='Frame')
ax2.set_xlabel('x'); ax2.set_ylabel('y')
ax2.set_title('§8 Player trajectory\n(physics + drag, 3s @ 60fps)')
ax2.legend(fontsize=8)

# Component table
ax3 = axes[2]; ax3.axis('off')
ax3.set_title('§8 Component registry\n(ECS data table)', fontsize=10)
comp_data = [
    ['Transform', 'x,y,angle,scale', 'All movable entities'],
    ['Velocity',  'vx,vy,angular',   'Physics-driven'],
    ['Health',    'hp,max_hp',       'Damageable'],
    ['Sprite',    'texture,w,h',     'Renderable'],
    ['Collider',  'radius,mask',     'Physics/collision'],
    ['AIBehavior','state,target',    'NPC logic'],
    ['TimedLife', 'duration',        'Particles/bullets'],
]
tbl3 = ax3.table(
    cellText=comp_data,
    colLabels=['Component','Fields','Used by'],
    loc='center', cellLoc='left')
tbl3.auto_set_font_size(False); tbl3.set_fontsize(8)
tbl3.scale(1,1.8)

plt.suptitle('§8: OOP Pygame Engine — ECS · Asset Manager · Physics · AI · Collision',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

print('\n§8 Engine stats:')
print(f'  Total entities: {len(world.em)}')
print(f'  Systems: {[type(s).__name__ for s in world.systems]}')
print(f'  Event subscribers: collision={len(world.bus._listeners.get("collision",[]))}')